In [1]:
import os, sys
os.chdir("/workspaces/Fintech/KhataSetu-Finance")
sys.path.append(os.getcwd())
os.makedirs("utils", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("pages", exist_ok=True)
print("cwd:", os.getcwd())

cwd: /workspaces/Fintech/KhataSetu-Finance


In [2]:
%%writefile utils/db.py
"""SQLite helpers for KhataSetu Finance."""

import sqlite3
import json
import os

DB_PATH = "data/khatasetu.db"


def get_connection():
    os.makedirs("data", exist_ok=True)
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def init_db():
    conn = get_connection()
    cur = conn.cursor()

    cur.execute("""
        CREATE TABLE IF NOT EXISTS borrowers (
            borrower_id INTEGER PRIMARY KEY AUTOINCREMENT,
            business_name TEXT NOT NULL,
            gstin TEXT,
            sector TEXT,
            cluster TEXT,
            turnover_band TEXT,
            language TEXT,
            base_monthly_turnover REAL,
            cash_flow_score REAL,
            risk_band TEXT,
            sanctioned_limit REAL,
            interest_rate_pct REAL,
            processing_fee_pct REAL,
            apr_pct REAL,
            raw_data_json TEXT,
            sub_scores_json TEXT,
            kyc_status TEXT DEFAULT 'pending',
            consent_given INTEGER DEFAULT 0,
            preferred_language TEXT,
            created_at TEXT DEFAULT (datetime('now'))
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS credit_offers (
            offer_id INTEGER PRIMARY KEY AUTOINCREMENT,
            borrower_id INTEGER NOT NULL,
            offered_limit REAL,
            interest_rate_pct REAL,
            processing_fee_pct REAL,
            apr_pct REAL,
            status TEXT DEFAULT 'offered',
            offered_at TEXT DEFAULT (datetime('now')),
            responded_at TEXT,
            FOREIGN KEY (borrower_id) REFERENCES borrowers(borrower_id)
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS disbursements (
            disbursement_id INTEGER PRIMARY KEY AUTOINCREMENT,
            offer_id INTEGER NOT NULL,
            borrower_id INTEGER NOT NULL,
            disbursed_amount REAL,
            esign_status TEXT DEFAULT 'pending',
            disbursed_at TEXT,
            FOREIGN KEY (offer_id) REFERENCES credit_offers(offer_id),
            FOREIGN KEY (borrower_id) REFERENCES borrowers(borrower_id)
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS repayments (
            repayment_id INTEGER PRIMARY KEY AUTOINCREMENT,
            disbursement_id INTEGER NOT NULL,
            borrower_id INTEGER NOT NULL,
            emi_amount REAL,
            due_date TEXT,
            paid_date TEXT,
            status TEXT DEFAULT 'due',
            FOREIGN KEY (disbursement_id) REFERENCES disbursements(disbursement_id),
            FOREIGN KEY (borrower_id) REFERENCES borrowers(borrower_id)
        )
    """)

    conn.commit()
    conn.close()


def insert_borrower(flat_row: dict, raw_data: dict, sub_scores: dict) -> int:
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("""
        INSERT INTO borrowers (
            business_name, gstin, sector, cluster, turnover_band, language,
            base_monthly_turnover, cash_flow_score, risk_band, sanctioned_limit,
            interest_rate_pct, processing_fee_pct, apr_pct,
            raw_data_json, sub_scores_json
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        flat_row.get("business_name"), flat_row.get("gstin"), flat_row.get("sector"),
        flat_row.get("cluster"), flat_row.get("turnover_band"), flat_row.get("language"),
        flat_row.get("base_monthly_turnover"), flat_row.get("cash_flow_score"),
        flat_row.get("risk_band"), flat_row.get("sanctioned_limit"),
        flat_row.get("interest_rate_pct"), flat_row.get("processing_fee_pct"),
        flat_row.get("apr_pct"), json.dumps(raw_data), json.dumps(sub_scores),
    ))
    conn.commit()
    borrower_id = cur.lastrowid
    conn.close()
    return borrower_id


def insert_borrowers_bulk(rows: list) -> list:
    """rows: list of (flat_row_dict, raw_data_dict, sub_scores_dict) tuples.
    Uses ONE connection for all inserts — fast."""
    conn = get_connection()
    cur = conn.cursor()
    inserted_ids = []

    for flat_row, raw_data, sub_scores in rows:
        cur.execute("""
            INSERT INTO borrowers (
                business_name, gstin, sector, cluster, turnover_band, language,
                base_monthly_turnover, cash_flow_score, risk_band, sanctioned_limit,
                interest_rate_pct, processing_fee_pct, apr_pct,
                raw_data_json, sub_scores_json
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            flat_row.get("business_name"), flat_row.get("gstin"), flat_row.get("sector"),
            flat_row.get("cluster"), flat_row.get("turnover_band"), flat_row.get("language"),
            flat_row.get("base_monthly_turnover"), flat_row.get("cash_flow_score"),
            flat_row.get("risk_band"), flat_row.get("sanctioned_limit"),
            flat_row.get("interest_rate_pct"), flat_row.get("processing_fee_pct"),
            flat_row.get("apr_pct"), json.dumps(raw_data), json.dumps(sub_scores),
        ))
        inserted_ids.append(cur.lastrowid)

    conn.commit()
    conn.close()
    return inserted_ids


def get_borrower(borrower_id: int) -> dict:
    conn = get_connection()
    row = conn.execute("SELECT * FROM borrowers WHERE borrower_id = ?", (borrower_id,)).fetchone()
    conn.close()
    if row is None:
        return None
    result = dict(row)
    result["raw_data"] = json.loads(result.pop("raw_data_json") or "{}")
    result["sub_scores"] = json.loads(result.pop("sub_scores_json") or "{}")
    return result


def get_all_borrowers(as_dataframe=False):
    conn = get_connection()
    rows = conn.execute("""
        SELECT borrower_id, business_name, gstin, sector, cluster, turnover_band,
               language, base_monthly_turnover, cash_flow_score, risk_band,
               sanctioned_limit, interest_rate_pct, processing_fee_pct, apr_pct,
               kyc_status, consent_given, created_at
        FROM borrowers
    """).fetchall()
    conn.close()
    result = [dict(r) for r in rows]
    if as_dataframe:
        import pandas as pd
        return pd.DataFrame(result)
    return result


def update_kyc_status(borrower_id: int, status: str):
    conn = get_connection()
    conn.execute("UPDATE borrowers SET kyc_status = ? WHERE borrower_id = ?", (status, borrower_id))
    conn.commit()
    conn.close()


def update_consent(borrower_id: int, given: bool):
    conn = get_connection()
    conn.execute("UPDATE borrowers SET consent_given = ? WHERE borrower_id = ?", (1 if given else 0, borrower_id))
    conn.commit()
    conn.close()


def create_offer(borrower_id: int, limit: float, rate: float, fee: float, apr: float) -> int:
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("""
        INSERT INTO credit_offers (borrower_id, offered_limit, interest_rate_pct, processing_fee_pct, apr_pct)
        VALUES (?, ?, ?, ?, ?)
    """, (borrower_id, limit, rate, fee, apr))
    conn.commit()
    offer_id = cur.lastrowid
    conn.close()
    return offer_id


def respond_to_offer(offer_id: int, status: str):
    conn = get_connection()
    conn.execute("UPDATE credit_offers SET status = ?, responded_at = datetime('now') WHERE offer_id = ?", (status, offer_id))
    conn.commit()
    conn.close()


def get_offers_for_borrower(borrower_id: int):
    conn = get_connection()
    rows = conn.execute("SELECT * FROM credit_offers WHERE borrower_id = ? ORDER BY offered_at DESC", (borrower_id,)).fetchall()
    conn.close()
    return [dict(r) for r in rows]


def create_disbursement(offer_id: int, borrower_id: int, amount: float) -> int:
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("INSERT INTO disbursements (offer_id, borrower_id, disbursed_amount) VALUES (?, ?, ?)", (offer_id, borrower_id, amount))
    conn.commit()
    disbursement_id = cur.lastrowid
    conn.close()
    return disbursement_id


def mark_esigned(disbursement_id: int):
    conn = get_connection()
    conn.execute("UPDATE disbursements SET esign_status = 'signed', disbursed_at = datetime('now') WHERE disbursement_id = ?", (disbursement_id,))
    conn.commit()
    conn.close()


def create_repayment_schedule(disbursement_id: int, borrower_id: int, emi_amount: float, due_dates: list):
    conn = get_connection()
    cur = conn.cursor()
    for due_date in due_dates:
        cur.execute("INSERT INTO repayments (disbursement_id, borrower_id, emi_amount, due_date) VALUES (?, ?, ?, ?)", (disbursement_id, borrower_id, emi_amount, due_date))
    conn.commit()
    conn.close()


def get_repayments_for_borrower(borrower_id: int):
    conn = get_connection()
    rows = conn.execute("SELECT * FROM repayments WHERE borrower_id = ? ORDER BY due_date", (borrower_id,)).fetchall()
    conn.close()
    return [dict(r) for r in rows]


def mark_repayment_paid(repayment_id: int):
    conn = get_connection()
    conn.execute("UPDATE repayments SET status = 'paid', paid_date = datetime('now') WHERE repayment_id = ?", (repayment_id,))
    conn.commit()
    conn.close()


def get_portfolio_summary() -> dict:
    conn = get_connection()
    total_borrowers = conn.execute("SELECT COUNT(*) FROM borrowers").fetchone()[0]
    total_sanctioned = conn.execute("SELECT SUM(sanctioned_limit) FROM borrowers").fetchone()[0] or 0
    total_disbursed = conn.execute("SELECT SUM(disbursed_amount) FROM disbursements").fetchone()[0] or 0
    npa_count = conn.execute("SELECT COUNT(*) FROM repayments WHERE status = 'npa'").fetchone()[0]
    band_counts = conn.execute("SELECT risk_band, COUNT(*) as count FROM borrowers GROUP BY risk_band").fetchall()
    conn.close()
    return {
        "total_borrowers": total_borrowers,
        "total_sanctioned": total_sanctioned,
        "total_disbursed": total_disbursed,
        "npa_count": npa_count,
        "band_distribution": {r["risk_band"]: r["count"] for r in band_counts},
    }

Overwriting utils/db.py


In [3]:
%%writefile utils/scoring_engine.py
"""Cash-flow based credit scorecard for KhataSetu Finance."""


def _score_gst(gst_returns: list) -> float:
    on_time_pct = sum(1 for r in gst_returns if r["filed_on_time"]) / len(gst_returns)
    avg_delay = sum(r["delay_days"] for r in gst_returns) / len(gst_returns)

    turnovers = [r["declared_turnover"] for r in gst_returns]
    avg_turnover = sum(turnovers) / len(turnovers)
    variance = sum((t - avg_turnover) ** 2 for t in turnovers) / len(turnovers)
    cv = (variance ** 0.5) / avg_turnover if avg_turnover else 1

    punctuality_score = on_time_pct * 100
    delay_penalty = min(avg_delay * 2, 30)
    stability_score = max(0, 100 - cv * 150)

    return max(0, min(100, 0.5 * punctuality_score - delay_penalty + 0.5 * stability_score))


def _score_upi(upi_history: list) -> float:
    inflows = [m["monthly_upi_inflow"] for m in upi_history]
    txn_counts = [m["txn_count"] for m in upi_history]

    trend_pct = (inflows[-1] - inflows[0]) / (inflows[0] + 1) * 100
    trend_score = max(0, min(100, 50 + trend_pct))

    avg_txn = sum(txn_counts) / len(txn_counts)
    depth_score = min(100, avg_txn / 8)

    return max(0, min(100, 0.6 * trend_score + 0.4 * depth_score))


def _score_eway(eway_bills: list) -> float:
    counts = [m["dispatch_count"] for m in eway_bills]
    momentum_pct = (counts[-1] - counts[0]) / (counts[0] + 1) * 100
    momentum_score = max(0, min(100, 50 + momentum_pct))

    avg_count = sum(counts) / len(counts)
    activity_score = min(100, avg_count / 2)

    return max(0, min(100, 0.5 * momentum_score + 0.5 * activity_score))


def _score_aa(aa_data: dict) -> float:
    credit = aa_data["avg_monthly_bank_credit"]
    debit = aa_data["avg_monthly_bank_debit"]
    balance = aa_data["avg_closing_balance"]
    bounces = aa_data["bounce_count_6m"]

    net_flow_ratio = (credit - debit) / credit if credit else 0
    net_flow_score = max(0, min(100, 50 + net_flow_ratio * 200))

    balance_ratio = balance / credit if credit else 0
    balance_score = min(100, balance_ratio * 300)

    bounce_penalty = bounces * 15

    return max(0, min(100, 0.5 * net_flow_score + 0.3 * balance_score - bounce_penalty + 20))


def _score_bureau(bureau: dict) -> float:
    if not bureau.get("has_bureau_file") or not bureau.get("bureau_score"):
        return 60
    return max(0, min(100, (bureau["bureau_score"] - 650) / 150 * 100))


def score_borrower(raw: dict) -> dict:
    gst_score = _score_gst(raw["gst_returns"])
    upi_score = _score_upi(raw["upi_history"])
    eway_score = _score_eway(raw["eway_bills"])
    aa_score = _score_aa(raw["aa_data"])
    bureau_score = _score_bureau(raw["bureau"])

    composite = round(
        0.30 * gst_score + 0.30 * upi_score + 0.15 * eway_score +
        0.20 * aa_score + 0.05 * bureau_score, 1
    )

    turnover = raw["base_monthly_turnover"]

    if composite >= 80:
        risk_band, limit_multiple, interest_rate, processing_fee_pct = "A", 3.0, 14.0, 1.0
    elif composite >= 65:
        risk_band, limit_multiple, interest_rate, processing_fee_pct = "B", 2.0, 18.0, 1.5
    elif composite >= 50:
        risk_band, limit_multiple, interest_rate, processing_fee_pct = "C", 1.0, 24.0, 2.0
    else:
        risk_band, limit_multiple, interest_rate, processing_fee_pct = "D", 0.4, 30.0, 2.5

    sanctioned_limit = round(turnover * limit_multiple, -3)
    apr = round(interest_rate + processing_fee_pct, 1)

    return {
        "score": composite,
        "risk_band": risk_band,
        "sanctioned_limit": sanctioned_limit,
        "interest_rate": interest_rate,
        "processing_fee_pct": processing_fee_pct,
        "apr": apr,
        "sub_scores": {
            "gst": round(gst_score, 1),
            "upi": round(upi_score, 1),
            "eway": round(eway_score, 1),
            "aa": round(aa_score, 1),
            "bureau": round(bureau_score, 1),
        },
    }

Overwriting utils/scoring_engine.py


In [4]:
%%writefile utils/synthetic_data.py
"""Generates synthetic GST / UPI / e-way bill / Account Aggregator data."""

import os
import random
import pandas as pd
from datetime import datetime, timedelta


def generate_borrower_data(seed=None):
    if seed is not None:
        random.seed(seed)

    base_monthly_turnover = random.randint(150000, 3500000)

    gst_returns = []
    for m in range(12):
        month_turnover = int(base_monthly_turnover * random.uniform(0.75, 1.25))
        filed_on_time = random.random() > 0.12
        gst_returns.append({
            "month": (datetime.now() - timedelta(days=30 * (11 - m))).strftime("%b %Y"),
            "declared_turnover": month_turnover,
            "filed_on_time": filed_on_time,
            "delay_days": 0 if filed_on_time else random.randint(1, 20),
        })

    upi_history = []
    for m in range(6):
        daily_inflow = int(base_monthly_turnover / 28 * random.uniform(0.7, 1.3))
        upi_history.append({
            "month": (datetime.now() - timedelta(days=30 * (5 - m))).strftime("%b %Y"),
            "monthly_upi_inflow": daily_inflow * 28,
            "txn_count": random.randint(150, 1200),
        })

    eway_bills = []
    for m in range(6):
        eway_bills.append({
            "month": (datetime.now() - timedelta(days=30 * (5 - m))).strftime("%b %Y"),
            "dispatch_count": random.randint(20, 300),
            "dispatch_value": int(base_monthly_turnover * random.uniform(0.5, 0.9)),
        })

    aa_data = {
        "avg_monthly_bank_credit": int(base_monthly_turnover * random.uniform(0.6, 1.0)),
        "avg_monthly_bank_debit": int(base_monthly_turnover * random.uniform(0.5, 0.95)),
        "avg_closing_balance": int(base_monthly_turnover * random.uniform(0.1, 0.4)),
        "bounce_count_6m": random.choices([0, 1, 2, 3], weights=[70, 15, 10, 5])[0],
    }

    bureau = {
        "has_bureau_file": random.random() > 0.55,
        "bureau_score": random.randint(650, 800) if random.random() > 0.55 else None,
    }

    return {
        "base_monthly_turnover": base_monthly_turnover,
        "gst_returns": gst_returns,
        "upi_history": upi_history,
        "eway_bills": eway_bills,
        "aa_data": aa_data,
        "bureau": bureau,
    }


CLUSTERS = ["UP - Moradabad", "Rajasthan - Bhilwara", "Gujarat - Morbi"]
SECTORS = ["Trading", "Micro-Manufacturing"]
TURNOVER_BANDS = ["< Rs 40L", "Rs 40L-1Cr", "Rs 1-3Cr", "Rs 3-5Cr", "> Rs 5Cr"]
LANGUAGES = ["Hindi", "English", "Gujarati", "Marwari"]

BUSINESS_PREFIXES = [
    "Sharma", "Gupta", "Patel", "Verma", "Singh", "Jain", "Yadav", "Mehta",
    "Chaudhary", "Reddy", "Bansal", "Malhotra", "Rathi", "Nair", "Iyer",
    "Agarwal", "Kapoor", "Shah", "Desai", "Pillai",
]
BUSINESS_SUFFIXES = [
    "Hardware Store", "Textiles", "Agri Traders", "Auto Parts", "Kirana Wholesale",
    "Food Processing", "Steel Traders", "Cloth House", "Fertilizer Depot",
    "Light Engineering", "Electricals", "Grains", "Ceramics", "Timber Traders",
    "Spice Traders", "Plastics", "Packaging Co", "Metal Works",
]


def _flatten_borrower_row(idx: int, raw: dict) -> dict:
    gst = raw["gst_returns"]
    upi = raw["upi_history"]
    eway = raw["eway_bills"]
    aa = raw["aa_data"]
    bureau = raw["bureau"]

    gst_turnovers = [r["declared_turnover"] for r in gst]
    gst_on_time_pct = round(100 * sum(1 for r in gst if r["filed_on_time"]) / len(gst), 1)
    gst_avg_delay_days = round(sum(r["delay_days"] for r in gst) / len(gst), 1)

    upi_inflows = [m["monthly_upi_inflow"] for m in upi]
    upi_trend_pct = round(100 * (upi_inflows[-1] - upi_inflows[0]) / (upi_inflows[0] + 1), 1)

    eway_counts = [m["dispatch_count"] for m in eway]
    eway_momentum_pct = round(100 * (eway_counts[-1] - eway_counts[0]) / (eway_counts[0] + 1), 1)

    return {
        "borrower_id": idx,
        "business_name": f"{random.choice(BUSINESS_PREFIXES)} {random.choice(BUSINESS_SUFFIXES)} #{idx}",
        "gstin": f"07ABCDE{1000+idx}F1Z{idx % 9}",
        "sector": random.choice(SECTORS),
        "cluster": random.choice(CLUSTERS),
        "turnover_band": random.choice(TURNOVER_BANDS),
        "language": random.choice(LANGUAGES),
        "base_monthly_turnover": raw["base_monthly_turnover"],
        "gst_filed_on_time_pct": gst_on_time_pct,
        "gst_avg_declared_turnover": round(sum(gst_turnovers) / len(gst_turnovers)),
        "gst_max_declared_turnover": max(gst_turnovers),
        "gst_min_declared_turnover":

Overwriting utils/synthetic_data.py


In [5]:
%%writefile utils/synthetic_data.py
"""Generates synthetic GST / UPI / e-way bill / Account Aggregator data."""

import os
import random
import pandas as pd
from datetime import datetime, timedelta


def generate_borrower_data(seed=None):
    if seed is not None:
        random.seed(seed)

    base_monthly_turnover = random.randint(150000, 3500000)

    gst_returns = []
    for m in range(12):
        month_turnover = int(base_monthly_turnover * random.uniform(0.75, 1.25))
        filed_on_time = random.random() > 0.12
        gst_returns.append({
            "month": (datetime.now() - timedelta(days=30 * (11 - m))).strftime("%b %Y"),
            "declared_turnover": month_turnover,
            "filed_on_time": filed_on_time,
            "delay_days": 0 if filed_on_time else random.randint(1, 20),
        })

    upi_history = []
    for m in range(6):
        daily_inflow = int(base_monthly_turnover / 28 * random.uniform(0.7, 1.3))
        upi_history.append({
            "month": (datetime.now() - timedelta(days=30 * (5 - m))).strftime("%b %Y"),
            "monthly_upi_inflow": daily_inflow * 28,
            "txn_count": random.randint(150, 1200),
        })

    eway_bills = []
    for m in range(6):
        eway_bills.append({
            "month": (datetime.now() - timedelta(days=30 * (5 - m))).strftime("%b %Y"),
            "dispatch_count": random.randint(20, 300),
            "dispatch_value": int(base_monthly_turnover * random.uniform(0.5, 0.9)),
        })

    aa_data = {
        "avg_monthly_bank_credit": int(base_monthly_turnover * random.uniform(0.6, 1.0)),
        "avg_monthly_bank_debit": int(base_monthly_turnover * random.uniform(0.5, 0.95)),
        "avg_closing_balance": int(base_monthly_turnover * random.uniform(0.1, 0.4)),
        "bounce_count_6m": random.choices([0, 1, 2, 3], weights=[70, 15, 10, 5])[0],
    }

    bureau = {
        "has_bureau_file": random.random() > 0.55,
        "bureau_score": random.randint(650, 800) if random.random() > 0.55 else None,
    }

    return {
        "base_monthly_turnover": base_monthly_turnover,
        "gst_returns": gst_returns,
        "upi_history": upi_history,
        "eway_bills": eway_bills,
        "aa_data": aa_data,
        "bureau": bureau,
    }


CLUSTERS = ["UP - Moradabad", "Rajasthan - Bhilwara", "Gujarat - Morbi"]
SECTORS = ["Trading", "Micro-Manufacturing"]
TURNOVER_BANDS = ["< Rs 40L", "Rs 40L-1Cr", "Rs 1-3Cr", "Rs 3-5Cr", "> Rs 5Cr"]
LANGUAGES = ["Hindi", "English", "Gujarati", "Marwari"]

BUSINESS_PREFIXES = [
    "Sharma", "Gupta", "Patel", "Verma", "Singh", "Jain", "Yadav", "Mehta",
    "Chaudhary", "Reddy", "Bansal", "Malhotra", "Rathi", "Nair", "Iyer",
    "Agarwal", "Kapoor", "Shah", "Desai", "Pillai",
]
BUSINESS_SUFFIXES = [
    "Hardware Store", "Textiles", "Agri Traders", "Auto Parts", "Kirana Wholesale",
    "Food Processing", "Steel Traders", "Cloth House", "Fertilizer Depot",
    "Light Engineering", "Electricals", "Grains", "Ceramics", "Timber Traders",
    "Spice Traders", "Plastics", "Packaging Co", "Metal Works",
]


def _flatten_borrower_row(idx: int, raw: dict) -> dict:
    gst = raw["gst_returns"]
    upi = raw["upi_history"]
    eway = raw["eway_bills"]
    aa = raw["aa_data"]
    bureau = raw["bureau"]

    gst_turnovers = [r["declared_turnover"] for r in gst]
    gst_on_time_pct = round(100 * sum(1 for r in gst if r["filed_on_time"]) / len(gst), 1)
    gst_avg_delay_days = round(sum(r["delay_days"] for r in gst) / len(gst), 1)

    upi_inflows = [m["monthly_upi_inflow"] for m in upi]
    upi_trend_pct = round(100 * (upi_inflows[-1] - upi_inflows[0]) / (upi_inflows[0] + 1), 1)

    eway_counts = [m["dispatch_count"] for m in eway]
    eway_momentum_pct = round(100 * (eway_counts[-1] - eway_counts[0]) / (eway_counts[0] + 1), 1)

    return {
        "borrower_id": idx,
        "business_name": f"{random.choice(BUSINESS_PREFIXES)} {random.choice(BUSINESS_SUFFIXES)} #{idx}",
        "gstin": f"07ABCDE{1000+idx}F1Z{idx % 9}",
        "sector": random.choice(SECTORS),
        "cluster": random.choice(CLUSTERS),
        "turnover_band": random.choice(TURNOVER_BANDS),
        "language": random.choice(LANGUAGES),
        "base_monthly_turnover": raw["base_monthly_turnover"],
        "gst_filed_on_time_pct": gst_on_time_pct,
        "gst_avg_declared_turnover": round(sum(gst_turnovers) / len(gst_turnovers)),
        "gst_max_declared_turnover": max(gst_turnovers),
        "gst_min_declared_turnover": min(gst_turnovers),
        "gst_avg_delay_days": gst_avg_delay_days,
        "upi_avg_monthly_inflow": round(sum(upi_inflows) / len(upi_inflows)),
        "upi_inflow_trend_pct": upi_trend_pct,
        "upi_avg_txn_count": round(sum(m["txn_count"] for m in upi) / len(upi)),
        "eway_avg_dispatch_count": round(sum(eway_counts) / len(eway_counts), 1),
        "eway_dispatch_momentum_pct": eway_momentum_pct,
        "eway_avg_dispatch_value": round(sum(m["dispatch_value"] for m in eway) / len(eway)),
        "aa_avg_monthly_credit": aa["avg_monthly_bank_credit"],
        "aa_avg_monthly_debit": aa["avg_monthly_bank_debit"],
        "aa_avg_closing_balance": aa["avg_closing_balance"],
        "aa_net_monthly_flow": aa["avg_monthly_bank_credit"] - aa["avg_monthly_bank_debit"],
        "aa_bounce_count_6m": aa["bounce_count_6m"],
        "bureau_has_file": bureau["has_bureau_file"],
        "bureau_score": bureau["bureau_score"] if bureau["bureau_score"] else "",
    }


def generate_dataset(n: int = 1000, seed: int = 42, score: bool = True) -> pd.DataFrame:
    from utils.scoring_engine import score_borrower
    random.seed(seed)
    rows = []
    scored_rows = []

    for i in range(1, n + 1):
        raw = generate_borrower_data()
        flat = _flatten_borrower_row(i, raw)
        rows.append(flat)

        if score:
            result = score_borrower(raw)
            scored_rows.append({
                "cash_flow_score": result["score"],
                "risk_band": result["risk_band"],
                "sanctioned_limit": result["sanctioned_limit"],
                "interest_rate_pct": result["interest_rate"],
                "processing_fee_pct": result["processing_fee_pct"],
                "apr_pct": result["apr"],
            })

    df = pd.DataFrame(rows)
    if score:
        df = pd.concat([df, pd.DataFrame(scored_rows)], axis=1)
    return df


def save_dataset_csv(path: str = "data/khatasetu_synthetic_dataset.csv", n: int = 1000, seed: int = 42):
    os.makedirs("data", exist_ok=True)
    df = generate_dataset(n=n, seed=seed)
    df.to_csv(path, index=False)
    return path


def load_dataset_to_db(n: int = 1000, seed: int = 42):
    """Fast bulk loader: builds all rows in memory, inserts with ONE connection."""
    from utils.scoring_engine import score_borrower
    from utils.db import init_db, insert_borrowers_bulk

    init_db()
    random.seed(seed)
    rows = []

    for i in range(1, n + 1):
        raw = generate_borrower_data()
        flat = _flatten_borrower_row(i, raw)
        result = score_borrower(raw)

        flat_row = {
            "business_name": flat["business_name"],
            "gstin": flat["gstin"],
            "sector": flat["sector"],
            "cluster": flat["cluster"],
            "turnover_band": flat["turnover_band"],
            "language": flat["language"],
            "base_monthly_turnover": flat["base_monthly_turnover"],
            "cash_flow_score": result["score"],
            "risk_band": result["risk_band"],
            "sanctioned_limit": result["sanctioned_limit"],
            "interest_rate_pct": result["interest_rate"],
            "processing_fee_pct": result["processing_fee_pct"],
            "apr_pct": result["apr"],
        }
        rows.append((flat_row, raw, result["sub_scores"]))

    return insert_borrowers_bulk(rows)


def reset_borrowers_table():
    """Wipes all borrower rows and resets the autoincrement counter."""
    from utils.db import get_connection
    conn = get_connection()
    conn.execute("DELETE FROM borrowers")
    conn.execute("DELETE FROM sqlite_sequence WHERE name='borrowers'")
    conn.commit()
    conn.close()

Overwriting utils/synthetic_data.py


In [9]:
for f in ["utils/db.py", "utils/scoring_engine.py", "utils/synthetic_data.py"]:
    with open(f) as file:
        content = file.read()
    print(f, "->", len(content), "chars")

import py_compile
for f in ["utils/db.py", "utils/scoring_engine.py", "utils/synthetic_data.py"]:
    py_compile.compile(f, doraise=True)
    print(f, "-> compiles OK")

utils/db.py -> 9954 chars
utils/scoring_engine.py -> 3885 chars
utils/synthetic_data.py -> 8116 chars
utils/db.py -> compiles OK
utils/scoring_engine.py -> compiles OK
utils/synthetic_data.py -> compiles OK


In [10]:
from utils.synthetic_data import reset_borrowers_table, load_dataset_to_db, save_dataset_csv

reset_borrowers_table()
path = save_dataset_csv()
ids = load_dataset_to_db(n=1000, seed=42)

print("CSV saved to:", path)
print(f"Inserted {len(ids)} borrowers, IDs {ids[0]}–{ids[-1]}")

CSV saved to: data/khatasetu_synthetic_dataset.csv
Inserted 1000 borrowers, IDs 1–1000


In [11]:
from utils.db import get_all_borrowers, get_portfolio_summary
df = get_all_borrowers(as_dataframe=True)
print(df.shape)
print(get_portfolio_summary())

(1000, 17)
{'total_borrowers': 1000, 'total_sanctioned': 3312165000.0, 'total_disbursed': 0, 'npa_count': 0, 'band_distribution': {'A': 89, 'B': 640, 'C': 270, 'D': 1}}
